# Multiclass Heart Disease Severity Classification

In [ ]:
# Imports
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    recall_score
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

In [ ]:
# Reproducibility Setup
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEEDS = [7, 21, 42]
set_seed(42)
print('Seeds:', SEEDS)

## Data Loading and Audit

In [ ]:
# Load and Audit Data
column_names = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
    'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'diagnosis'
]

candidate_paths = [
    Path('/home/rohanprashant/Downloads/heart+disease/processed.cleveland.data'),
    Path('/home/rohanprashant/Downloads/heart+disease/cleveland.data')
]

df = None
used_source = None
for p in candidate_paths:
    if p.exists():
        df = pd.read_csv(p, header=None, names=column_names)
        used_source = str(p)
        break

if df is None:
    backup_url = 'https://raw.githubusercontent.com/dataprofessor/data/master/heart-disease-cleveland.csv'
    df = pd.read_csv(backup_url)
    df.columns = [c.strip() for c in df.columns]
    used_source = backup_url

df = df.replace('?', np.nan)
for c in column_names:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

print('Data source:', used_source)
print('Raw shape:', df.shape)

print('Missing values by column:')
display(df.isna().sum().to_frame('missing').T)
print('Duplicate rows:', df.duplicated().sum())

df = df.dropna().copy()
df['diagnosis'] = df['diagnosis'].astype(int)

print('Clean shape:', df.shape)
print('Class distribution:')
display(df['diagnosis'].value_counts().sort_index().to_frame('count'))
df.head()

In [ ]:
# Split and Scale Data
feature_cols = [c for c in df.columns if c != 'diagnosis']
X = df[feature_cols].values
y = df['diagnosis'].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print('Train/Val/Test:', len(y_train), len(y_val), len(y_test))

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
X_val_t = torch.tensor(X_val_s, dtype=torch.float32)
X_test_t = torch.tensor(X_test_s, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_val_t = torch.tensor(y_val, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)

## Multiclass Neural Network

In [ ]:
# Define Multiclass Model and Utilities
class MultiNet(nn.Module):
    def __init__(self, input_dim, hidden_size, dropout, num_classes=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, num_classes)
        )

    def forward(self, x):
        return self.net(x)

def train_multiclass(model, X_tr, y_tr, X_v, y_v, lr, wd, epochs=140, batch_size=32):
    criterion = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)

    train_losses, val_losses = [], []
    for _ in range(epochs):
        model.train()
        total = 0.0
        for xb, yb in loader:
            opt.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            opt.step()
            total += loss.item() * len(xb)

        train_losses.append(total / len(X_tr))
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_v), y_v).item()
        val_losses.append(val_loss)

    return train_losses, val_losses

def predict_multiclass(model, X):
    model.eval()
    with torch.no_grad():
        logits = model(X)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        pred = probs.argmax(axis=1)
    return probs, pred

In [ ]:
# Hyperparameter Search
param_grid = {
    'lr': [1e-2, 1e-3],
    'hidden_size': [32, 64, 128],
    'dropout': [0.0, 0.3],
    'wd': [0.0, 1e-4]
}

input_dim = X_train_t.shape[1]
search_rows = []

for cfg in ParameterGrid(param_grid):
    seed_scores = []
    for s in SEEDS:
        set_seed(s)
        m = MultiNet(input_dim, cfg['hidden_size'], cfg['dropout'])
        train_multiclass(m, X_train_t, y_train_t, X_val_t, y_val_t, lr=cfg['lr'], wd=cfg['wd'])
        _, pred_val = predict_multiclass(m, X_val_t)
        seed_scores.append(f1_score(y_val, pred_val, average='macro'))

    row = dict(cfg)
    row['val_macro_f1_mean'] = float(np.mean(seed_scores))
    row['val_macro_f1_std'] = float(np.std(seed_scores))
    search_rows.append(row)

search_df = pd.DataFrame(search_rows).sort_values('val_macro_f1_mean', ascending=False).reset_index(drop=True)
display(search_df.head(8))
best_cfg = search_df.iloc[0].to_dict()
best_cfg

In [ ]:
# Multi-Seed Training and Selection
seed_metrics = []
seed_models = []

for s in SEEDS:
    set_seed(s)
    model = MultiNet(input_dim, int(best_cfg['hidden_size']), float(best_cfg['dropout']))
    tl, vl = train_multiclass(
        model, X_train_t, y_train_t, X_val_t, y_val_t,
        lr=float(best_cfg['lr']), wd=float(best_cfg['wd'])
    )

    probs_t, pred_t = predict_multiclass(model, X_test_t)
    seed_models.append({'seed': s, 'model': model, 'probs': probs_t, 'pred': pred_t, 'train_losses': tl, 'val_losses': vl})

    seed_metrics.append({
        'seed': s,
        'accuracy': accuracy_score(y_test, pred_t),
        'macro_f1': f1_score(y_test, pred_t, average='macro'),
        'weighted_f1': f1_score(y_test, pred_t, average='weighted')
    })

seed_metrics_df = pd.DataFrame(seed_metrics)
display(seed_metrics_df)
print('Mean +/- std')
display(seed_metrics_df.drop(columns=['seed']).agg(['mean', 'std']))

rep_seed = int(seed_metrics_df.sort_values('macro_f1', ascending=False).iloc[0]['seed'])
rep = [x for x in seed_models if x['seed'] == rep_seed][0]
y_pred = rep['pred']
print('Representative seed:', rep_seed)

In [ ]:
# Evaluate Multiclass Performance
acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average='macro')
weighted_f1 = f1_score(y_test, y_pred, average='weighted')
per_class_recall = recall_score(y_test, y_pred, average=None, labels=[0,1,2,3,4])
cm = confusion_matrix(y_test, y_pred, labels=[0,1,2,3,4])

print(f'Accuracy    : {acc:.4f}')
print(f'Macro F1    : {macro_f1:.4f}')
print(f'Weighted F1 : {weighted_f1:.4f}')
print('Per-class recall (0..4):', np.round(per_class_recall, 4))

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[0,1,2,3,4], yticklabels=[0,1,2,3,4])
plt.title('Multiclass Confusion Matrix (Test)')
plt.xlabel('Predicted class')
plt.ylabel('True class')
plt.tight_layout()
plt.show()

## Classical Baseline (Multinomial Logistic Regression)

In [ ]:
# Classical Baseline Comparison
lr_mc = LogisticRegression(max_iter=3000, multi_class='multinomial', random_state=42)
lr_mc.fit(X_train_s, y_train)
lr_pred = lr_mc.predict(X_test_s)

baseline = {
    'accuracy': accuracy_score(y_test, lr_pred),
    'macro_f1': f1_score(y_test, lr_pred, average='macro'),
    'weighted_f1': f1_score(y_test, lr_pred, average='weighted')
}

comparison = pd.DataFrame([
    {'model': 'NeuralNet', 'accuracy': acc, 'macro_f1': macro_f1, 'weighted_f1': weighted_f1},
    {'model': 'LogisticRegression', **baseline}
]).set_index('model')
display(comparison.round(4))

## Interpretability

In [ ]:
# Permutation Importance and Optional SHAP
perm = permutation_importance(
    lr_mc, X_test_s, y_test,
    scoring='f1_macro', n_repeats=20, random_state=42
)

perm_df = pd.DataFrame({
    'feature': feature_cols,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std
}).sort_values('importance_mean', ascending=False)
display(perm_df.head(10))

plt.figure(figsize=(8, 5))
sns.barplot(data=perm_df.head(10), x='importance_mean', y='feature', color='teal')
plt.title('Top 10 Permutation Importances (Multiclass Baseline)')
plt.tight_layout()
plt.show()

print('Optional SHAP (if installed):')
try:
    import shap
    explainer = shap.LinearExplainer(lr_mc, X_train_s)
    shap_values = explainer.shap_values(X_test_s)
    # For multiclass linear models, shap_values may be list-like.
    print('SHAP computed successfully.')
except Exception as e:
    print('SHAP unavailable or failed:', e)